In [3]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import os

In [4]:
FILE_PATH_LINKS = "proceduces_links.csv"
BASE_URL = "https://dichvucong.gov.vn/p/home/dvc-danh-sach-thu-tuc-theo-nhom-su-kien.html?objectId=1&groupEventId=753#mainTitle"
DRIVER_PATH = r"D:\chromedriver\chromedriver-win64\chromedriver.exe"
data = []

In [5]:
def make_driver(driver_path: str):
    options = webdriver.ChromeOptions()
    # options.add_argument("--headless")         
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-images")    
    options.add_argument("--blink-settings=imagesEnabled=false")
    s = Service(driver_path)

    return webdriver.Chrome(service=s, options=options)

In [6]:
def get_links(base_url: str, file_name: str):
    
    driver = make_driver(DRIVER_PATH)
    wait = WebDriverWait(driver, 10)
    total_links = 0
    links = []

    driver.get(base_url)
    ul = wait.until(EC.presence_of_element_located((By.ID, "ulTemplate")))
    items = ul.find_elements(By.TAG_NAME, "li")

    for item in items:
        li_tag = item.find_element(By.TAG_NAME, "a")
        link = li_tag.get_attribute("href")
        links.append(link)

    df = pd.DataFrame(links)
    df.to_csv(file_name, index=False, header=False, mode="a")   
    total_links += len(links)
    print(f"Đã xong, lấy được {len(links)} links")

    driver.quit()
    # print(f"Tổng số links {total_links}")

In [7]:
def get_links(base_url: str, file_name: str):
    driver = make_driver(DRIVER_PATH)
    wait = WebDriverWait(driver, 15)

    all_links = []
    seen = set()

    def extract_links_current_page():
        """Lấy tất cả link ở page hiện tại"""
        ul = wait.until(EC.presence_of_element_located((By.ID, "ulTemplate")))
        items = ul.find_elements(By.TAG_NAME, "li")

        page_links = []
        for item in items:
            try:
                a_tag = item.find_element(By.TAG_NAME, "a")
                link = a_tag.get_attribute("href")
                if link and link not in seen:
                    seen.add(link)
                    page_links.append(link)
            except Exception:
                continue

        return page_links

    try:
        driver.get(base_url)

        # --- Trang 1 ---
        page_1_links = extract_links_current_page()
        all_links.extend(page_1_links)
        print(f"Trang 1: lấy được {len(page_1_links)} links")

        # --- Trang 2, 3 ---
        for page_num in [2, 3]:
            try:
                # Lưu phần tử cũ để chờ DOM thay đổi
                old_first_link = wait.until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "#ulTemplate li a"))
                )

                # Tìm nút phân trang theo số trang
                page_btn = wait.until(
                    EC.element_to_be_clickable(
                        (
                            By.XPATH,
                            f"//a[normalize-space()='{page_num}'] | "
                            f"//button[normalize-space()='{page_num}'] | "
                            f"//*[self::a or self::button or self::span][normalize-space()='{page_num}']"
                        )
                    )
                )

                # Click bằng JS để tránh lỗi element not clickable
                driver.execute_script("arguments[0].click();", page_btn)

                # Chờ phần tử cũ bị stale => danh sách đã reload / re-render
                wait.until(EC.staleness_of(old_first_link))

                # Chờ danh sách mới xuất hiện lại
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#ulTemplate li a")))

                page_links = extract_links_current_page()
                all_links.extend(page_links)
                print(f"Trang {page_num}: lấy được {len(page_links)} links")

            except TimeoutException:
                print(f"Không tìm được hoặc không click được trang {page_num}")
                break

        # Ghi file 1 lần ở cuối
        df = pd.DataFrame(all_links)
        df.to_csv(file_name, index=False, header=False)

        print(f"Đã xong, tổng số links lấy được: {len(all_links)}")

    finally:
        driver.quit()

In [ ]:
def parse_table(table):

    headers = []
    try:
        thead = table.find_element(By.TAG_NAME, "thead")
        if thead:
            headers = [th.get_attribute("textContent").strip() for th in thead.find_elements(By.TAG_NAME, "th")]
    except:
        return []
    
    rows = []
    try:
        tbody = table.find_element(By.TAG_NAME, "tbody")
        if tbody:
            for tr in tbody.find_elements(By.TAG_NAME, "tr"):
                cells = [td.get_attribute("textContent").strip() for td in tr.find_elements(By.TAG_NAME, "td")]
                if cells and len(cells) == len(headers):
                    row_dict = dict(zip(headers, cells))
                    rows.append(row_dict)
    except:
        pass
    return rows

In [76]:
def parse_report_components(container):

    result = {}
    col_sm9 = container.find_element(By.CSS_SELECTOR, ".col-sm-9")
    children = col_sm9.find_elements(By.XPATH, "./*") 

    current_key = "Mặc định"  

    for child in children:
        tag = child.tag_name
        class_attr = child.get_attribute("class") or ""

        if tag == "div" and "key" in class_attr:
            current_key = child.get_attribute("textContent").strip()

        elif tag == "table":
            rows = parse_table(child)
            result[current_key] = rows

    return result

In [79]:
import json

# BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
# RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")

RAW_DATA_DIR = os.path.join("data", "raw")

os.makedirs(RAW_DATA_DIR, exist_ok=True)

def parse_listing(links: list[str]):
    driver = make_driver(DRIVER_PATH)

    for link in links:
        driver.get(link)

        a_tag = driver.find_element(By.CLASS_NAME, "url")
        driver.get(a_tag.get_attribute("href"))
        
        wait = WebDriverWait(driver, 10)

        attributes = wait.until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".info-row"))
        )
        procedure = {}
        for attribute in attributes:
            try:
                key = attribute.find_element(By.CSS_SELECTOR, ".col-sm-3.col-xs-12.key").get_attribute("textContent").strip()
                list_key_table = ["Cách thức thực hiện:", "Căn cứ pháp lý:"]
                value_element = attribute.find_element(By.CSS_SELECTOR, ".col-sm-9")
                value = value_element.get_attribute("textContent").strip()

                if key in list_key_table:
                    procedure[key] = parse_table(attribute)
                elif key == "Thành phần hồ sơ:":
                    procedure[key] = parse_report_components(attribute)
                else:
                    procedure[key] = value

                if key == "Mã thủ tục:":
                    PROCEDUCES_PATH = f"{value}.json"

            except Exception as e:
                print("Lỗi tại row:", attribute.get_attribute("outerHTML"))
                print("Error:", e)
            
        
        with open(PROCEDUCES_PATH, "a", encoding="utf-8") as file:
            json.dump(procedure, file, indent=4, ensure_ascii=False)
    driver.quit()
    

In [80]:
links = ["https://dichvucong.gov.vn/p/home/dvc-chi-tiet-thu-tuc-hanh-chinh.html?ma_thu_tuc=1.004194"]
# get_links(BASE_URL, FILE_PATH_LINKS)
parse_listing(links)

In [82]:
from docx import Document

doc = Document("1.MuCT01banhnhkmtheoThngts53.doc")
for para in doc.paragraphs:
    print(para.text)

KeyError: "no relationship of type 'http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument' in collection"